# 11 — Línea base global (XGBoost + MLP)

Entrena modelos **globales** sobre la unión de los clústeres LCA (`clase_0..6.parquet`)
replicando los mismos splits 70/15/15 por clúster (semilla 42) que los modelos por clúster,
de modo que la comparación de R²/RMSE/**MBE por territorio** (Tabla 4) sea válida.

Corre dos variantes:
1. `--mode global`: modelo global sin indicadoras.
2. `--mode global_indicators`: modelo global + indicadoras one-hot de `clase_lca` (territorio).

**Configuración del notebook:** acelerador **GPU T4 x2** (`NvidiaTeslaT4`) e Internet activado.
El dataset `lucasiturriago/pruebassaber11` se resuelve automáticamente (adjunto o vía `kagglehub`).

Salidas en `/kaggle/working/results/`: `metrics_global*.json`, `tabla4.csv`
(filas Global / Área / Clúster), `split_meta_global*.json` y `mlp_history_global*.json`.
Los modelos quedan en `/kaggle/working/models/`.

In [ ]:
!git clone https://github.com/liturriago/EquidadAlgoritmica-SaberXAI.git
%cd EquidadAlgoritmica-SaberXAI
!pip install -e . -q

In [ ]:
# Resolver el dataset: adjunto en /kaggle/input o descarga con kagglehub (fallback)
from pathlib import Path

cands = [
    Path('/kaggle/input/datasets/lucasiturriago/pruebassaber11'),
    Path('/kaggle/input/pruebassaber11'),
]
data_dir = next((p for p in cands if (p / 'clase_0.parquet').exists()), None)
if data_dir is None:
    import kagglehub
    data_dir = Path(kagglehub.dataset_download('lucasiturriago/pruebassaber11'))
print('data_dir =', data_dir)

In [ ]:
# Modelo global SIN indicadoras de territorio
!python scripts/train.py --config configs/global_baseline.yaml --mode global --data-dir {data_dir}

In [ ]:
# Modelo global CON indicadoras one-hot de territorio (clase_lca)
!python scripts/train.py --config configs/global_baseline.yaml --mode global_indicators --data-dir {data_dir}

In [ ]:
# Resumen: filas tipo Tabla 4 y metadatos del split
import pandas as pd
from IPython.display import display

tabla4 = pd.read_csv('results/tabla4.csv')
display(tabla4)

for tag in ('global', 'global_indicators'):
    display(pd.read_json(f'results/split_meta_{tag}.json', typ='series').to_frame(tag))